In [4]:
import os
import subprocess

WORKSPACE_ROOT = "/Users/elbethelzewdie/Downloads/Telegram Lite/el-workspace_new/el-workspace"  # CHANGE THIS
DATA = f"{WORKSPACE_ROOT}/data/split"
CONFIG = f"{WORKSPACE_ROOT}/config/config_nmt.yaml"

print(WORKSPACE_ROOT)

/Users/elbethelzewdie/Downloads/Telegram Lite/el-workspace_new/el-workspace


In [5]:
missing_files = []

for split in ["train", "dev", "test"]:
    en = f"{DATA}/{split}.en"
    am = f"{DATA}/{split}.am"

    if not os.path.exists(en):
        missing_files.append(en)
    if not os.path.exists(am):
        missing_files.append(am)

if missing_files:
    raise FileNotFoundError(f"Missing files:\n" + "\n".join(missing_files))

print("Dataset OK ✔")

Dataset OK ✔


In [6]:
!pip install subword-nmt

In [7]:
def run(cmd):
    subprocess.run(cmd, check=True)

def train_bpe(inp, out, size=32000):
    with open(inp, 'r', encoding='utf-8') as f_in, open(out, 'w', encoding='utf-8') as f_out:
        subprocess.run(["subword-nmt","learn-bpe","-s",str(size)], stdin=f_in, stdout=f_out, check=True)

def apply_bpe(inp, codes, out):
    with open(inp,'r',encoding='utf-8') as f_in, open(out,'w',encoding='utf-8') as f_out:
        subprocess.run(["subword-nmt","apply-bpe","-c",codes], stdin=f_in, stdout=f_out, check=True)

def build_vocab(inp, out):
    with open(inp,'r',encoding='utf-8') as f_in, open(out,'w',encoding='utf-8') as f_out:
        subprocess.run(["subword-nmt","get-vocab"], stdin=f_in, stdout=f_out, check=True)

In [8]:
# BPE training
en_bpe = f"{DATA}/bpe_codes.en"
am_bpe = f"{DATA}/bpe_codes.am"

if not os.path.exists(en_bpe):
    train_bpe(f"{DATA}/train.en", en_bpe)
if not os.path.exists(am_bpe):
    train_bpe(f"{DATA}/train.am", am_bpe)

print("BPE done")

100%|#########9| 31936/32000 [00:49<00:00, 530.47it/s] 

BPE done


100%|##########| 32000/32000 [00:49<00:00, 643.11it/s]


In [9]:
# Apply BPE
for split in ["train","dev","test"]:
    apply_bpe(f"{DATA}/{split}.en", f"{DATA}/bpe_codes.en", f"{DATA}/{split}.bpe.en")
    apply_bpe(f"{DATA}/{split}.am", f"{DATA}/bpe_codes.am", f"{DATA}/{split}.bpe.am")

print("Applied")

Applied


In [10]:
# Vocabulary
build_vocab(f"{DATA}/train.bpe.en", f"{DATA}/vocab.en")
build_vocab(f"{DATA}/train.bpe.am", f"{DATA}/vocab.am")

print("Vocab built")

Vocab built


In [12]:
TOKENIZER_LIB_DIR = f"{WORKSPACE_ROOT}/Tokenizer/build"
lib_path = f"{TOKENIZER_LIB_DIR}/libOpenNMTTokenizer.dylib"

if os.path.exists(lib_path):
    os.environ["DYLD_LIBRARY_PATH"] = TOKENIZER_LIB_DIR + ":" + os.environ.get("DYLD_LIBRARY_PATH", "")
    print("Tokenizer library loaded ✔")
else:
    print("❌ Tokenizer library NOT found at:", lib_path)

Tokenizer library loaded ✔


In [13]:
log_dir = f"{WORKSPACE_ROOT}/models/tensorboard"

subprocess.Popen(
    ["tensorboard", "--logdir", log_dir, "--port", "6006"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("TensorBoard running at http://localhost:6006")

TensorBoard running at http://localhost:6006


In [ ]:
subprocess.run([
    "onmt_train",
    "-config", CONFIG,
    "--tensorboard",
    "--tensorboard_log_dir", log_dir
], check=True)

[2026-04-20 21:37:14,786 INFO] Missing transforms field for train data, set to default: [].
[2026-04-20 21:37:14,786 WARNING] Corpus train's weight should be given. We default it to 1 for you.
[2026-04-20 21:37:14,786 INFO] Missing transforms field for valid data, set to default: [].
[2026-04-20 21:37:14,786 INFO] Parsed 2 corpora from -data.
[2026-04-20 21:37:14,786 INFO] Get special vocabs from Transforms: {'src': [], 'tgt': []}.
[2026-04-20 21:37:14,872 INFO] The first 10 tokens of the vocabs are:['<unk>', '<blank>', '<s>', '</s>', 'the', '.', 'of', '&apos;', 'to', 'and']
[2026-04-20 21:37:14,872 INFO] The decoder start token is: <s>
[2026-04-20 21:37:14,873 INFO] Building model...
[2026-04-20 21:37:15,673 INFO] Switching model to float32 for amp/apex_amp
[2026-04-20 21:37:15,673 INFO] Non quantized layer compute is fp32
[2026-04-20 21:37:15,674 INFO] NMTModel(
  (encoder): RNNEncoder(
    (embeddings): Embeddings(
      (make_embedding): Sequential(
        (emb_luts): Elementwise(

In [ ]:
print("DONE")